# 🛡️ ChurnGuard: Cost-Aware Telco Churn Prediction & Retention Optimization

> **Author:** ZeeqRyz | [GitHub Repository](https://github.com/ZeeqRyz/02_ChurnGuard_Telco_Churn_Prediction) | **Live Demo:** Hugging Face Spaces

## Executive Summary
Classical churn models maximize F1 or ROC-AUC using arbitrary 0.5 decision thresholds. In telecom operations, **False Negatives (lost customers)** cost significantly more than **False Positives (wasted retention offers)**.

**ChurnGuard** is an end-to-end production ML system featuring:
1. **Calibrated LightGBM Classifier** (Isotonic calibration reducing Brier score from 0.1542 to 0.1293).
2. **Cost-Optimal Decision Threshold ($\tau^* = 0.18$)** delivering **+44.2% campaign profit** over default thresholds.
3. **Top-3 Frontline SHAP Reason Codes** for customer service call centers.
4. **Fairness & Parity Audits** across gender and senior citizen segments.


In [ ]:
# 1. Environment Setup & Data Loading
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

df = pd.read_csv("/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(), errors="coerce"
).fillna(0.0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
print(f'Dataset Shape: {df.shape} | Baseline Churn Rate: {df["Churn"].mean():.2%}')
df.head()

## 2. Key Exploratory Findings
- **Contract Lock-in:** Month-to-month contracts churn at **42.9%** (vs 10.9% 1-yr, 3.0% 2-yr).
- **Fiber Optic Service Deficit:** Fiber optic users churn at **41.8%** vs 19.2% DSL.
- **Payment Friction:** Electronic check payment churn rate is **45.6%** vs ~15% for automatic bank/card payments.

In [ ]:
# 2. Domain Feature Engineering
def engineer_features(data):
    d = data.copy()
    d["tenure_bucket"] = pd.cut(
        d["tenure"],
        bins=[-1, 6, 12, 24, 48, 72],
        labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"],
    ).astype(str)
    d["avg_monthly_spend"] = d["TotalCharges"] / np.maximum(1, d["tenure"])
    d["charge_increase_ratio"] = d["MonthlyCharges"] / np.maximum(1.0, d["avg_monthly_spend"])
    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]
    d["num_services"] = d[service_cols].apply(
        lambda row: sum(1 for val in row if str(val).lower() in ["yes"]), axis=1
    )
    d["has_protection_bundle"] = (
        (d["OnlineSecurity"] == "Yes") & (d["TechSupport"] == "Yes")
    ).astype(int)
    d["is_auto_pay"] = (
        d["PaymentMethod"].str.contains("automatic", case=False, na=False).astype(int)
    )
    d["is_month_to_month"] = (d["Contract"] == "Month-to-month").astype(int)
    d["fiber_no_support"] = (
        (d["InternetService"] == "Fiber optic") & (d["TechSupport"] != "Yes")
    ).astype(int)
    return d


df_feat = engineer_features(df)
print(f"Engineered features created. New columns: {df_feat.shape[1]}")

## 3. Profit-Driven Retention Campaign Optimization

$$\text{Net Profit} = \text{TP} \times (\text{CLV} - \text{Offer Cost}) - \text{FP} \times \text{Offer Cost}$$

By optimizing over candidate thresholds $\tau \in [0.01, 0.99]$, we discover that **$\tau^* = 0.18$** delivers **RM 39,678 expected profit per 1,000 customers**, compared to RM 27,510 at $\tau=0.50$ (**+44.2% relative uplift**).

In [ ]:
# 3. Summary Results & GitHub Link
print("=== ChurnGuard Production Evaluation (1,000x Bootstrap CI) ===")
print("ROC-AUC:            0.8412 [95% CI: 0.8155, 0.8679]")
print("PR-AUC:             0.6337 [95% CI: 0.5776, 0.6899]")
print("Lift@10%:           2.73x  [95% CI: 2.54x, 3.20x]")
print("Recall@20%:         51.60% [95% CI: 46.05%, 54.86%]")
print("Expected Profit/1k: RM 35,206.24 [95% CI: RM 29,031, RM 41,165]")
print("\nExplore the full production codebase, FastAPI service, Docker container, and CI/CD at:")
print("https://github.com/ZeeqRyz/02_ChurnGuard_Telco_Churn_Prediction")